In [1]:
import pandas as pd
import numpy as np
import holidays

In [2]:
ELEKTRIK_PATH = "C:/Users/kerem/Desktop/elektrikveri/birleşik_veri.csv"
HAVA_PATH     = "C:/Users/kerem/Desktop/elektrikveri/Hava/turkey_weather_hourly.csv"
CIKTI_PATH    = "birlesik_veri.csv"

In [3]:
SEHIR_NUFUS = {
    "istanbul"  : 15.9,
    "ankara"    : 5.8,
    "izmir"     : 4.4,
    "antalya"   : 2.7,
    "gaziantep" : 2.2,
    "samsun"    : 1.4,
    "erzurum"   : 0.8,
}

In [4]:
df_elec = pd.read_csv(ELEKTRIK_PATH)
df_weather = pd.read_csv(HAVA_PATH)

df_elec["Datetime"] = pd.to_datetime(df_elec["Datetime"])
df_weather["time"]  = pd.to_datetime(df_weather["time"])

In [5]:
toplam  = sum(SEHIR_NUFUS.values())
agirlik = {s: n / toplam for s, n in SEHIR_NUFUS.items()}

hava_degiskenleri = {
    "temperature_2m"       : "sicaklik",
    "apparent_temperature" : "hissedilen_sicaklik",
    "relative_humidity_2m" : "nem",
    "cloud_cover"          : "bulut_ortus",
    "wind_speed_10m"       : "ruzgar_hizi",
    "precipitation"        : "yagis",
}

weather_pivot = df_weather.pivot_table(
    index="time", columns="city",
    values=list(hava_degiskenleri.keys()), aggfunc="first"
)

df_hava_list = []
for orijinal, yeni_ad in hava_degiskenleri.items():
    mevcut = {s: w for s, w in agirlik.items()
              if s in weather_pivot[orijinal].columns}
    toplam_w = sum(mevcut.values())
    agirlikli = sum(
        weather_pivot[orijinal][s] * (w / toplam_w)
        for s, w in mevcut.items()
    )
    df_hava_list.append(agirlikli.rename(yeni_ad))

df_hava = pd.concat(df_hava_list, axis=1).reset_index().rename(columns={"time": "Datetime"})

In [6]:
df = df_elec.rename(columns={
    "Üretim Miktarı (MWh)"    : "uretim_mwh",
    "Tüketim Miktarı(MWh)"   : "tuketim_mwh",
    "Ortalama Fiyat (TL/MWh)" : "fiyat_tl",
}).merge(df_hava, on="Datetime", how="left")

In [7]:
yillar = range(
    df["Datetime"].dt.year.min(),
    df["Datetime"].dt.year.max() + 1
)
tr_tatiller = holidays.Turkey(years=yillar)

df["is_holiday"] = df["Datetime"].dt.date.map(
    lambda d: 1 if d in tr_tatiller else 0
)

In [8]:
df["saat"]          = df["Datetime"].dt.hour        # 0–23
df["haftanin_gunu"] = df["Datetime"].dt.dayofweek   # 0=Pzt … 6=Paz
df["ay"]            = df["Datetime"].dt.month        # 1–12
df["is_weekend"]    = (df["haftanin_gunu"] >= 5).astype(int)

In [9]:
df[list(hava_degiskenleri.values())] = df[list(hava_degiskenleri.values())].round(2)

kolon_sirasi = [
    "Datetime",
    "uretim_mwh", "tuketim_mwh", "fiyat_tl",
    "sicaklik", "hissedilen_sicaklik", "nem",
    "bulut_ortus", "ruzgar_hizi", "yagis",
    "saat", "haftanin_gunu", "ay",
    "is_weekend", "is_holiday",
]
df = df[kolon_sirasi]
df.to_csv(CIKTI_PATH, index=False, encoding="utf-8-sig")

In [10]:
df.head()

,Datetime,uretim_mwh,tuketim_mwh,fiyat_tl,sicaklik,hissedilen_sicaklik,nem,bulut_ortus,ruzgar_hizi,yagis,saat,haftanin_gunu,ay,is_weekend,is_holiday
0,2021-01-01 00:00:00,29488.11,29489.46,259.39,9.79,6.91,77.41,59.42,13.50,0.01,0,4,1,0,1
1,2021-01-01 01:00:00,28065.76,28067.11,234.13,9.88,6.58,75.14,34.92,15.98,0.01,1,4,1,0,1
2,2021-01-01 02:00:00,26527.08,26527.08,215.25,9.58,6.37,77.43,37.49,15.81,0.01,2,4,1,0,1
3,2021-01-01 03:00:00,25327.19,25327.19,219.24,9.55,6.26,78.35,41.92,16.13,0.01,3,4,1,0,1
4,2021-01-01 04:00:00,24719.72,24719.72,209.81,9.32,6.26,79.02,36.95,14.82,0.06,4,4,1,0,1


In [11]:
print("=" * 55)
print("VERİ BİRLEŞTİRME DOĞRULAMA RAPORU")
print("=" * 55)

# 1. Satır sayısı
print(f"\nSatır sayısı : {len(df)}")
print(f"Beklenen     : 43824")
if len(df) != 43824:
    print(f"FARK: {43824 - len(df)} satır eksik!")
else:
    print("Tamam")

# 2. Hava verisi NaN kontrolü
hava_cols = list(hava_degiskenleri.values())
nan_hava = df[hava_cols].isnull().sum()
if nan_hava.any():
    print(f"\nHava verisi NaN'ları (merge eksikliği):")
    print(nan_hava[nan_hava > 0])
    # Eksik saatleri ffill ile doldur
    df[hava_cols] = df[hava_cols].ffill().bfill()
    print("ffill/bfill ile dolduruldu")
else:
    print("\nHava verisi: NaN yok")

# 3. Enerji değişkenleri NaN kontrolü
print(f"\nFiyat NaN (interpolasyon öncesi): {df['fiyat_tl'].isnull().sum()}")
# Zamana bağlı interpolasyon için Datetime'ı geçici olarak indeks yapıyoruz
df.set_index("Datetime", inplace=True)
df["fiyat_tl"] = df["fiyat_tl"].interpolate(method="time")
df.reset_index(inplace=True) # İndeksi tekrar sütuna çeviriyoruz
print(f"Fiyat NaN (interpolasyon sonrası): {df['fiyat_tl'].isnull().sum()}")

# 4. Tatil ve hafta sonu oranları
print(f"\nTatil günü oranı  : %{df['is_holiday'].mean()*100:.1f}")
print(f"Hafta sonu oranı  : %{df['is_weekend'].mean()*100:.1f}")

# 5. Hava ağırlık toplamı kontrolü
agirlik_toplam = sum(agirlik.values())
print(f"\nHava ağırlık toplamı: {agirlik_toplam:.4f}  {'✓' if abs(agirlik_toplam-1)<0.001 else '⚠️'}")

# 6. Şehir ağırlıkları
print("\nŞehir ağırlıkları:")
for sehir, w in sorted(agirlik.items(), key=lambda x: -x[1]):
    print(f"  {sehir:<12}: %{w*100:.1f}")

print("\nDoğrulama tamamlandı")

# 7. Dosyayı güncelle (NaN düzeltmeleri varsa)
df.to_csv(CIKTI_PATH, index=False, encoding="utf-8-sig")
print(f"✓ {CIKTI_PATH} güncellendi")

VERİ BİRLEŞTİRME DOĞRULAMA RAPORU

Satır sayısı : 43824
Beklenen     : 43824
✓ Tamam

✓ Hava verisi: NaN yok

Fiyat NaN (interpolasyon öncesi): 17
Fiyat NaN (interpolasyon sonrası): 0

Tatil günü oranı  : %3.8
Hafta sonu oranı  : %28.6

Hava ağırlık toplamı: 1.0000  ✓

Şehir ağırlıkları:
  istanbul    : %47.9
  ankara      : %17.5
  izmir       : %13.3
  antalya     : %8.1
  gaziantep   : %6.6
  samsun      : %4.2
  erzurum     : %2.4

✓ Doğrulama tamamlandı
✓ birlesik_veri.csv güncellendi
